# Introduction

In this exercise, building on the previous Lab 1 exercise and the tutorial on PyTerrier, you will be further familiarising yourself with PyTerrier by deploying various retrieval approaches and evaluating their impact on retrieval performance, as well as learning how to conduct an experiment in IR, and how to analyse results.


We have provided a medium size Web dataset (a user agreement to access and use this dataset is required), on which you will conduct your experiments. It is a sample of a TREC Web test collection, of approx. 800k documents, with corresponding topics (i.e. queries) & relevance assessments (i.e. qrels). If you have signed the required user agreement (see Moodle), you would have been emailed a personal login/password to access an index for this collection via PyTerrier's `get_dataset()` function, using the dataset name “50pct” and your personal credentials.  DO NOT SHARE YOUR CREDENTIALS.

This is an *individual exercise*. Your work will be submitted through the Exercise 1 Quiz Instance available on Moodle. The Quiz asks you various questions, which you should answer based on the experiments you have conducted. To support you conducting your work, you are strongly encouraged to make the best use of the two dedicated lab sessions to this exercise.

To help you structure your experiments, the rest of this notebook describes the experiments you need to conduct in relation to four tasks. Once you conduct a task, you should answer the corresponding questions on the Exercise 1 Quiz instance. **Ensure that you click the “Next Page” button to incrementally save your answers on the Quiz instance.**

## Assumed Knowledge

This Exercise assumes knowedge of Pandas and PyTerrier from Lab 1, which you should have completed by now (also see PyTerrier Tutorial). The relevant parts of the PyTerrier documentation are:
 - [Using Terrier indices in PyTerrier](https://pyterrier.readthedocs.io/en/latest/terrier-indexing.html)
 - [Terrier Retrieval using PyTerrier](https://pyterrier.readthedocs.io/en/latest/terrier-retrieval.html), e.g. `index.bm25()`
 - [Operators on PyTerrier transformers](https://pyterrier.readthedocs.io/en/latest/operators.html)




# Setup

NB: Windows users may need to use `%pip install  --user python-terrier gensim` -- you can ignore warnings about cython, PATH etc. If in doubt, resort to Colab

In [1]:
%pip install -q git+https://github.com/cmacdonald/irhm.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.3/208.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.0/149.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 56.4 MB/s eta 0:00

In [2]:
import pyterrier as pt

import pandas as pd
pd.set_option('display.max_colwidth', 150)
pd.set_option('display.max_rows', 200)

# Datasets for Ex1

For Exercise 1, we'll be using the Datasets API to obtain the files we need for this exercise. PyTerrier actually provides many datasets. You can list all of them using `pt.list_datasets()`.

In [3]:
pt.list_datasets().head()

,dataset,topics,topics_lang,qrels,corpus,corpus_lang,index,info_url
0,antique,"[train, test]",en,"[train, test]",True,en,None,https://ciir.cs.umass.edu/downloads/Antique/readme.txt
1,vaswani,True,en,True,True,en,True,http://ir.dcs.gla.ac.uk/resources/test_collections/npl/
2,msmarco_document,"[train, dev, test, test-2020, leaderboard-2020]",en,"[train, dev, test, test-2020]",True,en,True,https://microsoft.github.io/msmarco/
3,msmarcov2_document,"[train, dev1, dev2, valid1, valid2, trec_2021]",en,"[train, dev1, dev2, valid1, valid2]",None,None,True,https://microsoft.github.io/msmarco/TREC-Deep-Learning.html
4,msmarco_passage,"[train, dev, dev.small, eval, eval.small, test-2019, test-2020]",en,"[train, dev, test-2019, test-2020, dev.small]",True,en,True,https://microsoft.github.io/MSMARCO-Passage-Ranking/


There are several sets of files we need for Exercise 1:
 - We need an index for 50% of the TREC GOV corpus. We provide this through the "50pct" dataset, but you will need the username and password that you have been assigned once you accepted the user license agreement.
 - the topics (queries) and qrels (relevance assessments) for evaluating the performance of our search engine. These come from the "trec-wt-2004" dataset.

Update your username and password. DO NOT SHARE your login details with other students - all they need to do is to agree to the user agreement on Moodle.



In [4]:
USERNAME = "3090808s"
PASSWORD = "7f21809f"

dotgov_50pct = pt.get_dataset("irhm:50pct", user=USERNAME, password=PASSWORD)
dotgov_topicsqrels = pt.get_dataset("trec-wt-2004")

The size of the "50pct" index is 800MB - this will take a minute or so for Colab to download before we load it for the first time. You can read on while its downloading.

In [5]:
index_path = dotgov_50pct.get_index('ex2')
index = pt.terrier.TerrierIndex(index_path)

data.meta-0.fsomapfile:   0%|          | 0.00/50.1M [00:00<?, ?iB/s]

data.direct.bf:   0%|          | 0.00/263M [00:00<?, ?iB/s]

data.document.fsarrayfile:   0%|          | 0.00/13.1M [00:00<?, ?iB/s]

data.inverted.bf:   0%|          | 0.00/273M [00:00<?, ?iB/s]

data.lexicon.fsomapfile:   0%|          | 0.00/168M [00:00<?, ?iB/s]

data.lexicon.fsomaphash:   0%|          | 0.00/0.99k [00:00<?, ?iB/s]

data.lexicon.fsomapid:   0%|          | 0.00/7.80M [00:00<?, ?iB/s]

data.meta.idx:   0%|          | 0.00/6.16M [00:00<?, ?iB/s]

data.meta.zdata:   0%|          | 0.00/19.2M [00:00<?, ?iB/s]

data.properties:   0%|          | 0.00/4.13k [00:00<?, ?iB/s]

# Q1 [2 marks]

Using this setup, you now have sufficient knowledge from the Lab 1 to complete this task, namely to get the indexing statistics of the "50pct" collection.

Print the index collection statistics and answer the corresponding Quiz questions by entering the obtained indexing statistics: number of documents, number of terms, number of tokens, number of postings.


In [ ]:
# Q1: Index Collection Statistics
print("=" * 60)
print("INDEX COLLECTION STATISTICS")
print("=" * 60)

# Get collection statistics using the same approach as Lab 1
# The index.collection_statistics() returns a Java CollectionStatistics object
coll_stats = index.collection_statistics()

# Print all statistics
print("\nFull Collection Statistics:")
print(coll_stats)
print()

# Extract individual statistics using the Java object methods
num_documents = coll_stats.getNumberOfDocuments()
num_terms = coll_stats.getNumberOfUniqueTerms()
num_tokens = coll_stats.getNumberOfTokens()
num_postings = coll_stats.getNumberOfPointers()

print("=" * 60)
print("SUMMARY:")
print(f"Number of documents: {num_documents:,}")
print(f"Number of terms: {num_terms:,}")
print(f"Number of tokens: {num_tokens:,}")
print(f"Number of postings: {num_postings:,}")
print("=" * 60)

# Retrieval & Evaluation

In our experiments, we are using three sets of topics: homepage finding ("hp"), named page finding ("np") and topic distillation ("td"). They correspond to different user information needs on the Web:

- Homepage finding: The user's aim is to find the homepage of a given entity (person, organisation, etc) - e.g.  ‘University of Glasgow’, and the system should return the URL of that site’s homepage at (or near) rank one.

- Named page finding: The user aims to find a particular webpage/document - e.g. 'Uk Tax return form’, and the system should return the URL of that page at (or near) rank one.

- Topic distillation: The user aims to find as many relevant webpages as possible about a general topic. - e.g. ‘electoral  college’,  the  system  should  return and rank highly as many relevant webpages about the topic as possible. Each topic might have many relevant documents (similar to an adhoc search task).


For instance, to load the topics for "hp", you can do the following:

In [7]:
topics = dotgov_topicsqrels.get_topics(variant="hp")
topics.head(5)

Web2004.query.stream.trecformat.txt:   0%|          | 0.00/3.90k [00:00<?, ?iB/s]

04.topic-map.official.txt:   0%|          | 0.00/547 [00:00<?, ?iB/s]

,qid,query
0,6,Philadelphia streets
1,7,Togo embassy
2,9,Baltimore
3,17,Secure linux
4,29,Grand canyon monitoring and research center



Let's create a simple TF_IDF retriever - we will use this for demonstrating IR evaluation using PyTerrier.

In [8]:
retr = index.tf_idf()
retr

terrier-assemblies 5.11 jar-with-dependencies not found, downloading to /root/.pyterrier...


https://repo1.maven.org/maven2/org/terrier/terrier-assemblies/5.11/terrier-assemblies-5.11-jar-with-dependenci…

Done
terrier-python-helper 0.0.8 jar not found, downloading to /root/.pyterrier...


https://repo1.maven.org/maven2/org/terrier/terrier-python-helper/0.0.8/terrier-python-helper-0.0.8.jar:   0%| …

Done


Java started (triggered by Retriever.__init__) and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


TerrierRetr(/root/.pyterrier/corpora/50pct/index/ex2,{'terrierql': 'on', 'parsecontrols': 'on', 'parseql': 'on', 'applypipeline': 'on', 'localmatching': 'on', 'filters': 'on', 'decorate': 'on', 'wmodel': 'TF_IDF', 'tf_idf.k_1': 1.2, 'tf_idf.b': 0.75, 'decorate_batch': 'on', 'end': '999'},{'querying.processes': 'terrierql:TerrierQLParser,parsecontrols:TerrierQLToControls,parseql:TerrierQLToMatchingQueryTerms,matchopql:MatchingOpQLParser,applypipeline:ApplyTermPipeline,context_wmodel:org.terrier.python.WmodelFromContextProcess,localmatching:LocalManager$ApplyLocalMatching,qe:QueryExpansion,labels:org.terrier.learning.LabelDecorator,filters:LocalManager$PostFilterProcess,decorate:SimpleDecorateProcess', 'querying.postfilters': 'decorate:SimpleDecorate,site:SiteFilter,scope:Scope', 'querying.default.controls': 'wmodel:DPH,parsecontrols:on,parseql:on,applypipeline:on,terrierql:on,localmatching:on,filters:on,decorate:on', 'querying.allowed.controls': 'scope,qe,qemodel,start,end,site,scope,applypipeline', 'termpipelines': 'Stopwords,PorterStemmer'})

Let's see how we can actually evaluate our TF_IDF retrieval system. Firstly, we'll need the qrels.

In [9]:
qrels = dotgov_topicsqrels.get_qrels(variant='hp')

04.qrels.web.mixed.txt: 0.00iB [00:00, ?iB/s]

We can use `pt.Evaluate(results, qrels, metrics)` to evaluate the results. The metrics argument (with default value `["map", ndcg"]`) allows to configure the evaluation measures. For example, we can obtain the Mean Average Precision for a set of results `res` using relevance assessments `qrels` as follows:

In [10]:
res = retr.transform(topics)
eval = pt.Evaluate(res, qrels, metrics=["map"])
eval

{'map': 0.20894845478512017}

However, creating the `res` dataframe for each system in turn, and then evaluating it, is laborious and imperative in nature. We strongly recommend using [`pt.Experiment()`](https://pyterrier.readthedocs.io/en/latest/experiments.html) to evaluate one or more retrieval systems at once, in a declarative manner (see PyTerrier Tutorial). This also allows for significance testing.

Take the time to read the [documentation for `pt.Experiment()`](https://pyterrier.readthedocs.io/en/latest/experiments.html) to understand its available functionality. Tasks Q2-Q4 will all require that you adapt the arguments to `pt.Experiment()` and use its output in different ways (e.g. for significance testing).

In [11]:
pt.Experiment(
    [retr],
     dotgov_topicsqrels.get_topics(variant='hp'),
     dotgov_topicsqrels.get_qrels(variant='hp'),
     eval_metrics=['map']
)

,name,map
0,TerrierRetr(TF_IDF),0.208948


# Q2

Now you will experiment with three weighting models (TF_IDF, BM25 and PL2) and analyse their results on 3 different topic sets, representing different Web retrieval tasks: homepage finding (variant “hp”), named page finding (“np”), and topic distillation (“td”). These three topic sets and the corresponding qrels can also be accessed through PyTerrier's `get_dataset()` function, e.g.:

```python
topicsHP = pt.get_dataset("trec-wt-2004").get_topics("hp")
qrelsHP = pt.get_dataset("trec-wt-2004").get_qrels("hp")
```

In particular, we would like to compare the performances of the more advanced BM25 and PL2 term weighting models to those of TF_IDF, which is our baseline here. In other words, do the BM25 and PL2 models significantly improve the performances of the TF_IDF baseline on the three used topic sets?


# Q2(a)    [12 marks]

Provide the required MAP performances of each of the weighting models over the 3 topic sets. In particular, for each topic set (hp, np, td), compare the MAP performances of BM25 and PL2 to that of TF_IDF used as a *baseline* and answer if there are any observed statistical significance differences (p-value < 0.05) between the 3 models, when prompted by the Quiz. Next, provide the average MAP performance of each weighting model across the three topic sets (i.e. a global average performance over the three topic sets), when prompted by the Quiz instance. **Report your MAP performances rounded to 4 decimal places.**


*Hint*: We encourage you to write your own functions that perform reusable operations across different topic sets.

In [ ]:
# Q2(a): Compare TF_IDF, BM25, and PL2 on three topic sets

# Create retrieval systems
tf_idf = index.tf_idf()
bm25 = index.bm25()
pl2 = index.pl2()

retrievers = [tf_idf, bm25, pl2]
retriever_names = ["TF_IDF", "BM25", "PL2"]

# Topic sets
topic_sets = ["hp", "np", "td"]
topic_names = {"hp": "Homepage Finding", "np": "Named Page Finding", "td": "Topic Distillation"}

# Store results for each topic set
all_results = {}

print("=" * 80)
print("Q2(a): WEIGHTING MODEL COMPARISON")
print("=" * 80)

for variant in topic_sets:
    print(f"\n{'='*80}")
    print(f"Topic Set: {topic_names[variant]} ({variant})")
    print(f"{'='*80}")
    
    # Get topics and qrels
    topics = dotgov_topicsqrels.get_topics(variant=variant)
    qrels = dotgov_topicsqrels.get_qrels(variant=variant)
    
    # Run experiment with significance testing
    # baseline_0 means TF_IDF (index 0) is the baseline
    results = pt.Experiment(
        retrievers,
        topics,
        qrels,
        eval_metrics=["map"],
        names=retriever_names,
        baseline=0  # TF_IDF as baseline
    )
    
    all_results[variant] = results
    print(results)
    print()

# Calculate average MAP across the three topic sets
print("\n" + "=" * 80)
print("AVERAGE MAP ACROSS ALL TOPIC SETS")
print("=" * 80)

avg_maps = {}
for i, name in enumerate(retriever_names):
    maps = [all_results[variant].iloc[i]['map'] for variant in topic_sets]
    avg_map = sum(maps) / len(maps)
    avg_maps[name] = avg_map
    print(f"{name}: {avg_map:.4f}")

print("=" * 80)

#Q2(b)  [10 marks]

Next, for each topic set (hp, np, td), draw a single recall-precision graph comparing the performances of the 3 used weighting models (TF_IDF, BM25, PL2). Upload the resulting graphs into the Moodle instance when prompted (**check the graphs are readable/complete**). Then, answer the corresponding questions on the Quiz instance.


Hints:
 - You will need to use the `"iprec_at_recall"` measure, which gives precision at a given standard recall value.
 - Matplotlib has a [`savefig()`](https://chartio.com/resources/tutorials/how-to-save-a-plot-to-a-file-using-matplotlib/#the-savefig-method) function for saving a PNG of a figure.

In [ ]:
# Q2(b): Recall-Precision Graphs for each topic set

import matplotlib.pyplot as plt

# Standard recall levels for interpolated precision
recall_levels = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

print("=" * 80)
print("Q2(b): RECALL-PRECISION GRAPHS")
print("=" * 80)

for variant in topic_sets:
    print(f"\nGenerating recall-precision graph for {topic_names[variant]} ({variant})...")
    
    # Get topics and qrels
    topics = dotgov_topicsqrels.get_topics(variant=variant)
    qrels = dotgov_topicsqrels.get_qrels(variant=variant)
    
    # Get interpolated precision at recall for all models
    results = pt.Experiment(
        retrievers,
        topics,
        qrels,
        eval_metrics=["iprec_at_recall"],
        names=retriever_names
    )
    
    # Create figure
    plt.figure(figsize=(10, 7))
    
    # Plot each model
    for i, name in enumerate(retriever_names):
        precisions = []
        for recall in recall_levels:
            col_name = f"iprec_at_recall_{recall:.1f}"
            precisions.append(results.iloc[i][col_name])
        
        plt.plot(recall_levels, precisions, marker='o', label=name, linewidth=2)
    
    plt.xlabel('Recall', fontsize=12)
    plt.ylabel('Precision', fontsize=12)
    plt.title(f'Recall-Precision Curve - {topic_names[variant]}', fontsize=14)
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    
    # Save figure
    filename = f'recall_precision_{variant}.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"  Saved: {filename}")
    plt.show()

print("\n" + "=" * 80)

#Q2 (c) [1 mark]

Finally, you should now answer on the Quiz the most effective weighting model (in terms of average Mean Average Precision), which you will use for the rest of Exercise 1. To find this model, simply identify the weighting model with the highest average performance over the 3 topic sets.  


In [ ]:
# Q2(c): Identify the best weighting model

print("=" * 80)
print("Q2(c): BEST WEIGHTING MODEL")
print("=" * 80)

# Find the model with highest average MAP
best_model = max(avg_maps, key=avg_maps.get)
best_map = avg_maps[best_model]

print(f"\nBest Weighting Model: {best_model}")
print(f"Average MAP: {best_map:.4f}")
print(f"\nThis model will be used for the remaining exercises (Q3 and beyond).")
print("=" * 80)

# Store the best retriever for use in Q3
if best_model == "TF_IDF":
    best_retriever = tf_idf
elif best_model == "BM25":
    best_retriever = bm25
else:  # PL2
    best_retriever = pl2

# Q3 Query Expansion

Query expansion is one of the most well-known and effective techniques for improving the effectiveness of a search engine. We'll be using the Terrier's Bo1 query expansion model, which is a Pseudo-Relevance Feedback (PRF) method.

You will now conduct the Query Expansion experiments using the weighting model that produced the highest average Mean Average Precision (MAP) across the 3 topic sets in Q2.

See the [relevant documentation](https://pyterrier.readthedocs.io/en/latest/rewrite.html#bo1queryexpansion) about creating a QE transformer pipeline in PyTerrier using the [`bo1` method](https://pyterrier.readthedocs.io/en/latest/terrier/api.html#pyterrier.terrier.TerrierIndex.bo1).





#Q3(a).  [6 marks]

For each of the topic sets (i.e. homepage finding (hp), named page finding (np), and topic distillation (td)), run an experiment evaluating the application of query expansion on the best weighting model identified in Q2(c) used here as the *baseline*. Query expansion has a few parameters, e.g. query expansion model, number of documents to analyse, number of expansion terms - in conducting your experiments, you should simply use the default query expansion settings of Terrier: Bo1, 3 documents, 10 expansion terms. Report the obtained MAP performances in the Quiz instance then answer the corresponding questions. **Report your MAP performances rounded to 4 decimal places.** Include a screenshot of the [schematic](https://pyterrier.readthedocs.io/en/latest/schematic.html) of the query expansion pipeline you use.

Recall that the required experiments for evaluating the application of query expansion should be conducted with the best weighting model identified in the previous question Q2(c).

In [ ]:
# Q3(a): Query Expansion with Bo1

print("=" * 80)
print("Q3(a): QUERY EXPANSION WITH Bo1")
print("=" * 80)

# Create query expansion pipeline using Bo1
# Default parameters: 3 documents, 10 expansion terms
qe_pipeline = best_retriever >> index.bo1()

print(f"\nQuery Expansion Pipeline: {best_model} >> Bo1 (fb_docs=3, fb_terms=10)")
print("\nSchematic of Query Expansion Pipeline:")
print(qe_pipeline)
print()

# Visualize the pipeline schematic
try:
    from pyterrier.schematic import schematic
    import IPython.display as display
    
    diagram = schematic(qe_pipeline)
    display.display(diagram)
except:
    print("(Schematic visualization requires additional dependencies)")

# Store QE results for each topic set
qe_results = {}

for variant in topic_sets:
    print(f"\n{'='*80}")
    print(f"Topic Set: {topic_names[variant]} ({variant})")
    print(f"{'='*80}")
    
    # Get topics and qrels
    topics = dotgov_topicsqrels.get_topics(variant=variant)
    qrels = dotgov_topicsqrels.get_qrels(variant=variant)
    
    # Compare baseline vs QE
    results = pt.Experiment(
        [best_retriever, qe_pipeline],
        topics,
        qrels,
        eval_metrics=["map"],
        names=[f"{best_model} (Baseline)", f"{best_model} + Bo1 QE"],
        baseline=0
    )
    
    qe_results[variant] = results
    print(results)
    print()

print("=" * 80)

#Q3(b)   [9 marks]

Now, you will delve into the performance of the best retrieval model identified in Question Q2(c) using the topic distillation (“td”) topic set. Draw a query-delta bar chart (see example in Lecture 5) comparing the Average Precision (AP) performance of your system with and without query expansion - each bar represents the difference in average precision for a given query between the baseline and after applying QE for that query (i.e. “delta_AP = QE_AP - noQE_AP”). As the topic set has 75 queries, your figure should only show the queries that have delta_AP > 0.02 absolute, in order to focus on queries that have the biggest positive or negative changes. Your x-axis should be labelled with the qid and the text of the original query and your queries should be ordered as could be seen in Lecture 5 (Slide 62). Using the produced bar chart (**check the graph is readable/complete**), and the corresponding data, you should now be able to answer the corresponding questions in the Quiz.

*Hints*:
 - You will need to use the `perquery=True` option for `pt.Experiment()`. You will also need to analyse the expanded queries.
 - You may need a [self-join](https://www.w3schools.com/sql/sql_join_self.asp) on a dataframe.
 - You can iterate through a dataframe using [`dataframe.iterrows()`](https://cmdlinetips.com/2018/12/how-to-loop-through-pandas-rows-or-how-to-iterate-over-pandas-rows/)
 - You can examine the expanded queries by adjusting your pipeline, and executing the pipeline on the relevant topics.

In [ ]:
# Q3(b): Query-Delta Bar Chart for Topic Distillation

print("=" * 80)
print("Q3(b): QUERY-DELTA ANALYSIS FOR TOPIC DISTILLATION")
print("=" * 80)

# Get topics and qrels for TD
topics_td = dotgov_topicsqrels.get_topics(variant='td')
qrels_td = dotgov_topicsqrels.get_qrels(variant='td')

# Run per-query evaluation
per_query_results = pt.Experiment(
    [best_retriever, qe_pipeline],
    topics_td,
    qrels_td,
    eval_metrics=["ap"],
    names=["Baseline", "QE"],
    perquery=True
)

# Calculate delta AP for each query
# Merge baseline and QE results
baseline_results = per_query_results[per_query_results['name'] == 'Baseline'][['qid', 'ap']].copy()
qe_results_df = per_query_results[per_query_results['name'] == 'QE'][['qid', 'ap']].copy()

# Rename columns for clarity
baseline_results.columns = ['qid', 'ap_baseline']
qe_results_df.columns = ['qid', 'ap_qe']

# Merge the dataframes
delta_df = baseline_results.merge(qe_results_df, on='qid')
delta_df['delta_ap'] = delta_df['ap_qe'] - delta_df['ap_baseline']

# Add query text
delta_df = delta_df.merge(topics_td[['qid', 'query']], on='qid')

# Filter queries with |delta_AP| > 0.02
significant_deltas = delta_df[abs(delta_df['delta_ap']) > 0.02].copy()

# Sort by delta_ap
significant_deltas = significant_deltas.sort_values('delta_ap', ascending=True)

print(f"\nTotal queries: {len(delta_df)}")
print(f"Queries with |delta_AP| > 0.02: {len(significant_deltas)}")
print(f"Queries improved (delta_AP > 0.02): {len(significant_deltas[significant_deltas['delta_ap'] > 0])}")
print(f"Queries degraded (delta_AP < -0.02): {len(significant_deltas[significant_deltas['delta_ap'] < 0])}")

# Create bar chart
if len(significant_deltas) > 0:
    plt.figure(figsize=(16, 8))
    
    # Create x-axis labels (qid: query)
    labels = [f"{row['qid']}: {row['query'][:50]}..." if len(row['query']) > 50 
              else f"{row['qid']}: {row['query']}" 
              for _, row in significant_deltas.iterrows()]
    
    # Create bar chart
    colors = ['green' if x > 0 else 'red' for x in significant_deltas['delta_ap']]
    bars = plt.bar(range(len(significant_deltas)), significant_deltas['delta_ap'], color=colors, alpha=0.7)
    
    plt.axhline(y=0, color='black', linestyle='-', linewidth=0.8)
    plt.xticks(range(len(significant_deltas)), labels, rotation=90, ha='right', fontsize=8)
    plt.ylabel('Delta AP (QE - Baseline)', fontsize=12)
    plt.xlabel('Query ID and Text', fontsize=12)
    plt.title('Query-Level Impact of Query Expansion on Topic Distillation\n(Showing queries with |Δ AP| > 0.02)', fontsize=14)
    plt.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    
    # Save figure
    filename = 'query_delta_td.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"\nSaved: {filename}")
    plt.show()
    
    # Display statistics
    print("\n" + "=" * 80)
    print("DELTA AP STATISTICS")
    print("=" * 80)
    print(f"Mean Delta AP: {delta_df['delta_ap'].mean():.4f}")
    print(f"Median Delta AP: {delta_df['delta_ap'].median():.4f}")
    print(f"Std Dev Delta AP: {delta_df['delta_ap'].std():.4f}")
    print(f"Max improvement: {delta_df['delta_ap'].max():.4f}")
    print(f"Max degradation: {delta_df['delta_ap'].min():.4f}")
else:
    print("\nNo queries with |delta_AP| > 0.02 found.")

print("=" * 80)

# Q4 Word Embeddings [10 marks]

Next, implement a new query expansion method as a PyTerrier transformer that uses a Word2Vec model for identifying semantically related terms. In particular, your implementation should take each query term in an incoming query, identify the most semantically related words - using a provided Word2Vec model and the Gensim Python library - to add to the query. In conducting this experiment, you need to choose:

-  How many similar terms to identify for each existing query term so as to ensure a *fair comparison* with the experiments conducted in Q3;

-  The relative importance of these new terms compared to the existing query terms;

-  How/if to integrate the Word2Vec cosine distance into your weighting formula.

-  An adequate pipeline to use your custom Word2Vec QE transformer.  

Compare the performance of your model in comparison to the **PL2 baseline** on the *topic distillation (“td”)* topic set. How many queries are improved or are degraded? Check how your new query expansion mechanism compares to Terrier’s standard Bo1 query expansion mechanism.


## Background on word2vec

Q4 asks for a word2vec-based query expansion model. Word2vec (also called word embeddings) is a shallow neural network where semantically similar words end up with similar embedding vectors.

If you haven't taken Text-as-Data, you can do some background reading about word embeddings at:
 - https://towardsdatascience.com/introduction-to-word-embedding-and-word2vec-652d0c2060fa
 - https://en.wikipedia.org/wiki/Word2vec
 - https://en.wikipedia.org/wiki/Word_embedding

In general, while word2vec is still a very widely used model, note that it has been surpassed by more complex models such as BERT. But word2vec is still useful to consider in the context of query expansion.


# Setup of Gensim

In this part of the exercise, we will use [Gensim](https://radimrehurek.com/gensim/auto_examples/tutorials/run_word2vec.html), a Python toolkit for working with a word2vec model.

We are providing a pre-trained word2vec model that Gensim will download and open - the file is quite large, so this might take a few minutes to download and a couple of minutes to load. You can read on while it opens.

In [ ]:
import gensim.downloader as api
%time model = api.load("glove-wiki-gigaword-300")

[=============-------------------------------------] 27.8% 104.7/376.1MB downloaded

# Example Usage of Gensim

`model` is of type [gensim.models.keyedvectors.KeyedVectors](https://radimrehurek.com/gensim/models/keyedvectors.html#gensim.models.keyedvectors.KeyedVectors).

You can think of this as a dictionary mapping string words to the vector embeddings for each word.  For example, we can get the vector for the word `'government'` as follows:

In [ ]:
emb = model.get_vector("government")
print(emb.shape)
print(emb)

As you can see, each word is represented by a 300-dimension vector.

We can also ask `model` for the most similar words to `'government'` using [`model.most_similar()`](https://radimrehurek.com/gensim/models/keyedvectors.html#gensim.models.keyedvectors.KeyedVectors.most_similar). It returns the 10 most similar words, based on the cosine similarity of their emebddings to that of `'government'`.

See also: [Example in Gensim documentation](https://radimrehurek.com/gensim/models/keyedvectors.html#what-can-i-do-with-word-vectors).

In [ ]:
model.most_similar("government")

As you can see, some words are clearly related to the original word `'government'`, including some lexical variations (`'governments'`), as well as semantically similar (`"authorities"`) words. You can also see some words that perhaps seem unrelated - probably they are highly weighted because they appeared in similar contexts to `"government"` (e.g. `"saying"`).

# Now develop your Word2Vec-based Query Expansion

The next task is to use `model` to develop your custom transformer for a word2vec-based query expansion, and use it with PL2.

*Hints about the custom transformer*:
 - Inspired by Pandas, PyTerrier has the notion of [apply functions](https://pyterrier.readthedocs.io/en/latest/apply.html) for applying transformations.
 - What to do with out-of-vocabulary (OOV) words?
 - How many similar terms to identify for each existing query term?
 - How to ensure fair comparison with the experiments conducted in Q3.
 - What is the relative importance of these new terms compared to the existing query terms? e.g. you might want to give more emphasis to the original query terms (See Lecture 6).
 - How/if to integrate the Word2Vec cosine distance into your weighting formula?
 - How to deal with special characters not recognised by the default Terrier query parser, causing a QueryParserException (e.g `/`)?

*Hints about integration*:
 - Think *very* carefully about the required pipeline to use your custom word2vec-based query expansion transformer. It should *not* be used in the same way as Bo1. If your pipeline is very slow, this might be the problem.
 - You will be required to upload the schematic visualisation for your pipeline

Recall from the start of Q4 that you need to compare the performance of your QE model in comparison to the PL2 baseline on the topic distillation (“td”) topic set.

You now have sufficient information to make a start on Q4. In the Quiz instance, insert your source code for your PyTerrier transformer (note that a **2-bands penalty** will be applied if you do not upload the code you used to answer the Quiz questions of Q4). Next, answer the remaining questions in the Quiz. **Ensure that your notebook shows evidence of all work you have done to answer all of the Q4 Quiz questions or marks will be lost.**

In [ ]:
# Q4: Word2Vec-based Query Expansion

import re

print("=" * 80)
print("Q4: WORD2VEC-BASED QUERY EXPANSION")
print("=" * 80)

# Custom Word2Vec Query Expansion Transformer
class Word2VecQE(pt.Transformer):
    """
    Word2Vec-based Query Expansion Transformer
    
    Parameters:
    - model: Gensim Word2Vec model
    - num_terms_per_word: Number of similar terms to find for each query term
    - original_weight: Weight boost for original query terms (default: 2.0)
    - use_similarity_weights: Whether to weight expansion terms by cosine similarity
    """
    
    def __init__(self, model, num_terms_per_word=2, original_weight=2.0, use_similarity_weights=True):
        super().__init__()
        self.model = model
        self.num_terms_per_word = num_terms_per_word
        self.original_weight = original_weight
        self.use_similarity_weights = use_similarity_weights
    
    def transform(self, topics_df):
        """Transform topics by expanding queries with Word2Vec similar terms"""
        import copy
        output_df = topics_df.copy()
        expanded_queries = []
        
        for idx, row in topics_df.iterrows():
            original_query = row['query']
            query_terms = original_query.lower().split()
            
            # Build weighted query
            query_parts = []
            expansion_terms = []
            
            # Add original terms with higher weight
            for term in query_terms:
                # Clean term
                clean_term = re.sub(r'[^\w\s]', '', term)
                if clean_term:
                    query_parts.append(f"{clean_term}^{self.original_weight}")
                    
                    # Find similar terms using Word2Vec
                    try:
                        similar = self.model.most_similar(clean_term, topn=self.num_terms_per_word)
                        for sim_word, sim_score in similar:
                            # Clean similar word (remove special characters that cause QueryParserException)
                            clean_sim = re.sub(r'[^\w\s]', '', sim_word)
                            if clean_sim and clean_sim not in query_terms:
                                if self.use_similarity_weights:
                                    # Weight by similarity score
                                    weight = round(sim_score, 2)
                                    expansion_terms.append(f"{clean_sim}^{weight}")
                                else:
                                    expansion_terms.append(clean_sim)
                    except KeyError:
                        # Term not in vocabulary - skip
                        pass
            
            # Combine original and expansion terms
            expanded_query = " ".join(query_parts + expansion_terms)
            expanded_queries.append(expanded_query)
        
        output_df['query'] = expanded_queries
        return output_df

print("\nWord2Vec Query Expansion Transformer Implementation:")
print("=" * 80)
print("""
Key Design Decisions:
1. num_terms_per_word=2: To ensure fair comparison with Bo1 (10 total expansion terms)
   - With ~5 query terms, 2 terms per word ≈ 10 expansion terms total
2. original_weight=2.0: Original terms weighted 2x higher than expansion terms
3. use_similarity_weights=True: Weight expansion terms by Word2Vec cosine similarity
4. OOV handling: Skip terms not in Word2Vec vocabulary
5. Special character removal: Prevent QueryParserException errors
""")
print("=" * 80)

# Create Word2Vec QE transformer
# For fair comparison with Bo1 (which uses 10 expansion terms from 3 documents),
# we use 2 similar terms per query word
w2v_qe = Word2VecQE(model, num_terms_per_word=2, original_weight=2.0, use_similarity_weights=True)

# Create pipeline: Apply W2V QE first, then retrieve with PL2
# Important: W2V QE should be applied BEFORE retrieval (not after like Bo1)
pl2_baseline = index.pl2()
w2v_pipeline = w2v_qe >> pl2_baseline

print("\nWord2Vec QE Pipeline:")
print(w2v_pipeline)
print()

# Visualize pipeline schematic
try:
    from pyterrier.schematic import schematic
    import IPython.display as display
    
    print("Pipeline Schematic:")
    diagram = schematic(w2v_pipeline)
    display.display(diagram)
except:
    print("(Schematic visualization requires additional dependencies)")

# Also create Bo1 QE pipeline with PL2 for comparison
bo1_qe_pipeline = pl2_baseline >> index.bo1()

print("\n" + "=" * 80)
print("EVALUATION ON TOPIC DISTILLATION (TD)")
print("=" * 80)

# Get topics and qrels for TD
topics_td = dotgov_topicsqrels.get_topics(variant='td')
qrels_td = dotgov_topicsqrels.get_qrels(variant='td')

# Run experiment comparing PL2 baseline, Bo1 QE, and Word2Vec QE
results = pt.Experiment(
    [pl2_baseline, bo1_qe_pipeline, w2v_pipeline],
    topics_td,
    qrels_td,
    eval_metrics=["map"],
    names=["PL2 (Baseline)", "PL2 + Bo1 QE", "PL2 + Word2Vec QE"],
    baseline=0
)

print(results)

# Per-query analysis
print("\n" + "=" * 80)
print("PER-QUERY ANALYSIS: Word2Vec QE vs PL2 Baseline")
print("=" * 80)

per_query = pt.Experiment(
    [pl2_baseline, w2v_pipeline],
    topics_td,
    qrels_td,
    eval_metrics=["ap"],
    names=["Baseline", "W2V_QE"],
    perquery=True
)

# Calculate improvements and degradations
baseline_ap = per_query[per_query['name'] == 'Baseline'][['qid', 'ap']].copy()
w2v_ap = per_query[per_query['name'] == 'W2V_QE'][['qid', 'ap']].copy()

baseline_ap.columns = ['qid', 'ap_baseline']
w2v_ap.columns = ['qid', 'ap_w2v']

comparison = baseline_ap.merge(w2v_ap, on='qid')
comparison['delta'] = comparison['ap_w2v'] - comparison['ap_baseline']

improved = len(comparison[comparison['delta'] > 0])
degraded = len(comparison[comparison['delta'] < 0])
unchanged = len(comparison[comparison['delta'] == 0])

print(f"\nQueries improved: {improved}")
print(f"Queries degraded: {degraded}")
print(f"Queries unchanged: {unchanged}")
print(f"Total queries: {len(comparison)}")

print("\n" + "=" * 80)
print("COMPARISON: Word2Vec QE vs Bo1 QE")
print("=" * 80)

# Compare W2V vs Bo1
per_query_all = pt.Experiment(
    [pl2_baseline, bo1_qe_pipeline, w2v_pipeline],
    topics_td,
    qrels_td,
    eval_metrics=["ap"],
    names=["Baseline", "Bo1_QE", "W2V_QE"],
    perquery=True
)

baseline_df = per_query_all[per_query_all['name'] == 'Baseline'][['qid', 'ap']].rename(columns={'ap': 'ap_baseline'})
bo1_df = per_query_all[per_query_all['name'] == 'Bo1_QE'][['qid', 'ap']].rename(columns={'ap': 'ap_bo1'})
w2v_df = per_query_all[per_query_all['name'] == 'W2V_QE'][['qid', 'ap']].rename(columns={'ap': 'ap_w2v'})

full_comparison = baseline_df.merge(bo1_df, on='qid').merge(w2v_df, on='qid')
full_comparison['w2v_better_than_bo1'] = full_comparison['ap_w2v'] > full_comparison['ap_bo1']

w2v_wins = full_comparison['w2v_better_than_bo1'].sum()
bo1_wins = (~full_comparison['w2v_better_than_bo1']).sum()

print(f"Queries where Word2Vec QE outperforms Bo1 QE: {w2v_wins}")
print(f"Queries where Bo1 QE outperforms Word2Vec QE: {bo1_wins}")

print("\n" + "=" * 80)
print("SAMPLE EXPANDED QUERIES (First 5 queries)")
print("=" * 80)

# Show some example expanded queries
sample_topics = topics_td.head(5)
expanded_topics = w2v_qe.transform(sample_topics)

for i in range(len(sample_topics)):
    print(f"\nQuery {sample_topics.iloc[i]['qid']}:")
    print(f"  Original: {sample_topics.iloc[i]['query']}")
    print(f"  Expanded: {expanded_topics.iloc[i]['query']}")

print("\n" + "=" * 80)
print("Q4 COMPLETE")
print("=" * 80)

# That's all Folks

**Submission Instructions:** Complete this notebook, and answer the related questions in the Exercise 1 Quiz Instance on Moodle. As part of the Quiz, you will be asked to upload your .ipynb notebook (**showing both your solutions and the results of their execution**) and answer questions as per the exercise specification (use File... Download .ipynb).

**IMPORTANT:** Your notebook should indicate **clearly** how your code blocks correspond to each question. Please note that a **2-bands penalty** will be applied, if you do not upload your completed notebook or if you do not show all the results (including plots) obtained from the execution of your solutions. Your completed notebook **MUST** show both your solutions and the results of their executions. The submitted notebook will be used to *spot check* your answers in the Quiz. **Marks can be lost** if the notebook does not show evidence for the reported answers submitted in your Quiz. This exercise is worth 50 marks and 10% of the final course grade.


**NB:** Remember that you can (and should) naturally complete the answers to the quiz over several iterations. However, *please ensure that you save your intermediary work on the Quiz instance by clicking the “Next Page” button every time you make any change in a given page of the quiz and you want it to be saved*.

Your responses to the Quiz along with your ipynb notebook solution must be submitted by the stated deadline.